In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))
from src.utils import OPTIONS, clean, map_at_3, make_submission

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

In [3]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv('../data/test.csv')

In [4]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [6]:
train['question'] = train['prompt'].map(clean)
test['question'] = test['prompt'].map(clean)

for opt in OPTIONS:
    train[f'{opt}_clean'] = train[opt].map(lambda x : clean(x, is_prompt=False))
    test[f'{opt}_clean'] = test[opt].map(lambda x : clean(x, is_prompt=False))

In [7]:
def combined_format(df, has_labels=True):
    rows = []
    for _, row in df.iterrows():
        for opt in OPTIONS:
            rows.append({
                'id': row['id'],
                'option': opt,
                'combined': row['question'] + ' ' + row[f'{opt}_clean'],
                'label': int(row['answer'] == opt) if has_labels else None
            })
    return pd.DataFrame(rows)
 
train_tf = combined_format(train, has_labels=True)
test_tf = combined_format(test, has_labels=False)
 
print(train_tf.head(5))
train_tf['combined'][0]

   id option                                           combined  label
0   1      A  what is martin heidegger s view on the relatio...      0
1   1      B  what is martin heidegger s view on the relatio...      1
2   1      C  what is martin heidegger s view on the relatio...      0
3   1      D  what is martin heidegger s view on the relatio...      0
4   1      E  what is martin heidegger s view on the relatio...      0


'what is martin heidegger s view on the relationship between time and human existence martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end the relationship to the past involves acknowledging it as a historical era and the relationship to the future involves creating a world that will endure beyond one s own time'

In [8]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
 
X_train = vectorizer.fit_transform(train_tf['combined'])
y_train = train_tf['label'].values
 
X_test  = vectorizer.transform(test_tf['combined'])
 
X_train.shape

(10000, 12445)

In [9]:
groups = train_tf['id'].values
gkf = GroupKFold(n_splits=5)
 
fold_scores = []
 
model = LogisticRegression(max_iter=1000, C=1.0, class_weight={0:1, 1:3})

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    model.fit(X_train[tr_idx], y_train[tr_idx])
 
    val = train_tf.iloc[val_idx].copy()
    val['prob'] = model.predict_proba(X_train[val_idx])[:, 1]

    preds, truths = [], []
    for qid, group in val.groupby('id'):
        ranked = group.sort_values('prob', ascending=False)['option'].tolist()
        preds.append(ranked[:3])
        truths.append(train.loc[train['id'] == qid, 'answer'].values[0])
 
    score = map_at_3(truths, preds)
    fold_scores.append(score)
    print(f"Fold{fold} MAP@3: {score:.4f}")

Fold0 MAP@3: 0.9862
Fold1 MAP@3: 0.9900
Fold2 MAP@3: 0.9825
Fold3 MAP@3: 0.9862
Fold4 MAP@3: 0.9875


In [10]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,"{0: 1, 1: 3}"
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [11]:
test_tf['prob'] = model.predict_proba(X_test)[:, 1]
 
test_preds = []
test_ids = []
for qid, group in test_tf.groupby('id'):
    ranked = group.sort_values('prob', ascending=False)['option'].tolist()
    test_preds.append(ranked[:3])
    test_ids.append(qid)
 
submission = make_submission(test_ids, test_preds)
submission.to_csv('submission.csv', index=False)
print(submission.head())

   id prediction
0   1      A E B
1   2      B E A
2   3      B E D
3   4      E C A
4   5      C A B
